In [1]:
import re
import pandas as pd
import time
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns


### Reading in the Data

In [2]:
race = 'TOR450'

In [3]:
TOR450_dem = pd.read_excel(f'{race} Data/5. Clean Data for Data Visualisation/{race}_dem.xlsx')

In [4]:
TOR450_dem['Bib'][(TOR450_dem['Year'] == 2022) &
               (TOR450_dem['Retired'] == 'Bosses')].nunique()

0

In [5]:
TOR450_dem.groupby( ['Year','Status1'] )['Status1' ].count()

Year  Status1               
2021  Finished at Courmayeur    1
Name: Status1, dtype: int64

In [6]:
lifebase_cut_offs_df = pd.read_excel(f'{race} Data/4. TOR450 Timetable Data/{race}_lifebase_cut_offs_df.xlsx')

# # # Convert integer seconds to timedelta
lifebase_cut_offs_df['Lifebase Duration'] = pd.to_timedelta(lifebase_cut_offs_df['Lifebase Duration_seconds'], unit='s')

# # # Convert integer seconds to timedelta
lifebase_cut_offs_df['Running Total Lifebase Duration'] = pd.to_timedelta(lifebase_cut_offs_df['Running Total Lifebase Duration_seconds'], unit='s')

lifebase_cut_offs_df

,Lifebase,Lifebase Distance (km),Lifebase Accumulated Distance Elevation (m),Lifebase Elevation Gain (m),Lifebase Accumulated Elevation (m),Lifebase Duration_seconds,Running Total Lifebase Duration_seconds,Lifebase Duration,Running Total Lifebase Duration
0,Cogne IN,159,159,10844,10844,230400,230400,2 days 16:00:00,2 days 16:00:00
1,Donnas IN,68,227,4948,15792,90000,320400,1 days 01:00:00,3 days 17:00:00
2,Gressoney IN,60,287,5647,21439,118800,439200,1 days 09:00:00,5 days 02:00:00
3,FINISH,156,443,10878,32317,244800,684000,2 days 20:00:00,7 days 22:00:00


In [7]:
checkpoints_bib_df = pd.read_excel(f'{race} Data/5. Clean Data for Data Visualisation/{race}_checkpoints_bib_df.xlsx')

In [10]:
# all_aid_station_bib_df = pd.read_excel(f'{race} Data/5. Clean Data for Data Visualisation/{race}_all_aid_station_bib_df.xlsx')

In [9]:
lifebase_bib_df = pd.read_excel(f'{race} Data/5. Clean Data for Data Visualisation/{race}_lifebase_bib_df.xlsx')

In [12]:
# making duration hours 
datasets = [checkpoints_bib_df, lifebase_bib_df,
#             all_aid_station_bib_df,
            TOR450_dem]
for df in datasets:
    df['Duration_hours'] = df['Duration_seconds']/ 3600 

### Getting Average and Median time for lifebases

In [18]:
checkpoint_category_order = [ 'START',
                        'Cogne IN', 'Cogne OUT',
                        'Donnas IN', 'Donnas OUT',
                        'Gressoney IN', 'Gressoney OUT', 
                        'Rifugio Champillon', 
                             'FINISH']

lifebase_category_order = ['START',
            'Cogne IN', 'Cogne OUT', 
            'Donnas IN', 'Donnas OUT', 
            'Gressoney IN', 'Gressoney OUT', 
                           'FINISH' ]



Stage1 = [ 'START', 'Rifugio Maison Vieille', 'Rifugio Elisabetta',
 'Rifugio Deffeyes', 'Rifugio degli Angeli', 'Rifugio Bezzi',
 'Rifugio Benevolo', 'Rifugio Savoia', 'Rifugio Vittorio Emanuele II',
 'Rifugio Chabod', 'Rifugio Sella', 'Cogne IN']

Stage2 = [ 'Cogne OUT', 'Rifugio Grauson', 'Rifugio Dondena', 'Rifugio Miserin', 'Dortoir Crest',
 'Dortoir Retempio', 'Rifugio Bonze', 'Donnas IN']

Stage3 = [ 'Donnas OUT', 'Perloz','Sassa',  'Rifugio Coda', 'Rifugio della Barma',
 'Lago Chiaro', 'Col della Vecchia', 'Niel La Gruba',
 'Loo', 'Gressoney IN']

Stage4 =  ['Gressoney OUT', 'Rifugio Sitten', 'Rifugio Ferraro','Rifugio Guide di Frachey',
 'Rifugio Duca degli Abruzzi', 'Hotel Stambecco', 'Rifugio Perucca Vuillermoz',
 'Rifugio Prarayer', 'Rifugio Crête Sèche',
 'Rifugio Champillon', 'Ponteille Desot',
 'Hotel Italia', 'Rifugio Frassati',
 'Pas Entre Deux Sauts', 'Monte de la Saxe', 'Parco Bollino', 'FINISH'] 

Stage4_diversion =  [  'Oyace',  'Oyace OUT',
 'Bruson Arp', 'Col Brison',
 'Berio Damon', 'Ollomont IN',
 'Ollomont OUT']

DNF_areas =  [  'Champoluc', 'Valtourenche OUT', 'Bosses']

stages =[ Stage1, Stage2, Stage3, Stage4, Stage4_diversion, DNF_areas]
stages_str =[ 'Stage 1', 'Stage 2', 'Stage 3', 'Stage 4', 'Stage 4 Diversion', 'DNFed on TOR330 route' ]

lifebase_time_spent = ['Cogne OUT','Donnas OUT','Gressoney OUT']


In [24]:
def creating_time_stats(df, column, category_order):
    
    df_merge = df.merge(
        TOR450_dem[['PK', 'Finish Category']],
        on=['PK'],
        how='left')


    df_merge['Duration_seconds'][df_merge['Duration_seconds']  <= 0] = np.nan

    stats_df = df_merge.groupby(['Finish Category', column])['Duration_seconds'].describe().reset_index(drop = False)


    stats_df[[ 'mean', 'std', 'min', '25%',
           '50%', '75%', 'max']] = stats_df[[ 'mean', 'std', 'min', '25%',
           '50%', '75%', 'max']].round(0)

    # Set 'Finish Category' as a categorical column with the defined order
    stats_df[column] = pd.Categorical(
        stats_df[column],
        categories=category_order,
        ordered=True)
    
    stats_df = stats_df.sort_values(by=column, ascending = True)
    
    

    for finish_category in df_merge['Finish Category'].unique():
#         print(finish_category)
        stats_df.loc[
                (stats_df['Finish Category'] == finish_category), 'running_total_mean_seconds'
            ] =     stats_df.loc[
                (stats_df['Finish Category'] == finish_category), 'mean'
            ].cumsum()


        stats_df.loc[
                (stats_df['Finish Category'] == finish_category), 'running_total_median_seconds'
            ] =     stats_df.loc[
                (stats_df['Finish Category'] == finish_category), '50%'
            ].cumsum()


    stats_df = stats_df[['Finish Category', column,
            'count', 'mean', '50%', 'std', 'min', 'max', 
            'running_total_mean_seconds', 'running_total_median_seconds']]


    stats_df = stats_df.rename(columns={'count': f'Count_Finish_Category_{column}_seconds',
                                      'mean': f'Mean_Finish_Category_{column}_seconds',
                                      'std': f'STD_Finish_Category{column}t_seconds',
                                      '50%': f'Median_Finish_Category_{column}_seconds',
                                      'min': f'Min_Finish_Category_{column}_seconds', 
                                      'max': f'Max_Finish_Category_{column}_seconds'})
    stats_df = stats_df[stats_df[column] != 'Start']

    
    
    
    for stage,  stage_str in zip(stages, stages_str):
        stats_df.loc[stats_df[column].isin(stage), 'Stage'] = f'{stage_str}'
        
    for lifebase in lifebase_time_spent:
        lifebase_split = lifebase.split(' OUT')[0] 
        stats_df.loc[stats_df[column] == lifebase, 'Stage'] = f'Time Spent in {lifebase_split}'


    stats_df.to_excel(f'{race} Data/5. Clean Data for Data Visualisation/{race}_{column}_duration_mean_median_stats_df.xlsx', index = False)

#     print(stats_df[stats_df['Finish Category'] == 'Sub-130'])
    return stats_df

In [19]:
lifebase_stats_df = creating_time_stats(df = lifebase_bib_df, 
                    column = 'Lifebase', 
                    category_order = lifebase_category_order)

lifebase_stats_df

C:\Users\Karina\AppData\Local\Temp\ipykernel_14576\725103266.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merge['Duration_seconds'][df_merge['Duration_seconds']  <= 0] = np.nan


Sub-140
Sub-170
Sub-180
DNF
Sub-160
Sub-190
Sub-150
Sub-130
Sub-120


,Finish Category,Lifebase,Count_Finish_Category_Lifebase_seconds,Mean_Finish_Category_Lifebase_seconds,Median_Finish_Category_Lifebase_seconds,STD_Finish_CategoryLifebaset_seconds,Min_Finish_Category_Lifebase_seconds,Max_Finish_Category_Lifebase_seconds,running_total_mean_seconds,running_total_median_seconds,Stage
71,Sub-190,START,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stage 1
63,Sub-180,START,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stage 1
55,Sub-170,START,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stage 1
47,Sub-160,START,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stage 1
7,DNF,START,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stage 1
...,...,...,...,...,...,...,...,...,...,...,...
12,Sub-120,FINISH,2.0,154914.0,154914.0,8870.0,148642.0,161186.0,413595.0,413595.0,Stage 4
28,Sub-140,FINISH,10.0,184350.0,182730.0,4307.0,178531.0,191262.0,492385.0,488170.0,Stage 4
4,DNF,FINISH,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stage 4
60,Sub-180,FINISH,36.0,239235.0,235784.0,12197.0,224959.0,272349.0,632590.0,630705.0,Stage 4


In [33]:
# # # Convert integer seconds to timedelta
lifebase_stats_df['Median_Finish_Category_Lifebase'] = pd.to_timedelta(lifebase_stats_df['Median_Finish_Category_Lifebase_seconds'], unit='s')
lifebase_stats_df['Mean_Finish_Category_Lifebase'] = pd.to_timedelta(lifebase_stats_df['Mean_Finish_Category_Lifebase_seconds'], unit='s')


lifebase_stats_df[['Finish Category','Lifebase', 'Mean_Finish_Category_Lifebase', 'Median_Finish_Category_Lifebase']][lifebase_stats_df['Finish Category'] == 'Sub-190']

,Finish Category,Lifebase,Mean_Finish_Category_Lifebase,Median_Finish_Category_Lifebase
71,Sub-190,START,NaT,NaT
64,Sub-190,Cogne IN,2 days 04:07:31,2 days 04:00:24
65,Sub-190,Cogne OUT,0 days 03:23:37,0 days 03:20:22
66,Sub-190,Donnas IN,1 days 02:29:02,1 days 02:43:40
67,Sub-190,Donnas OUT,0 days 02:18:11,0 days 01:56:51
69,Sub-190,Gressoney IN,1 days 03:00:32,1 days 03:19:54
70,Sub-190,Gressoney OUT,0 days 02:30:47,0 days 02:13:32
68,Sub-190,FINISH,2 days 22:56:35,2 days 22:47:24


In [37]:
checkpoints_stats_df = creating_time_stats(df = checkpoints_bib_df, 
                    column = 'Checkpoint', 
                    category_order = checkpoint_category_order)



# # # Convert integer seconds to timedelta
checkpoints_stats_df['Median_Finish_Category_Checkpoint'] = pd.to_timedelta(checkpoints_stats_df['Median_Finish_Category_Checkpoint_seconds'], unit='s')
checkpoints_stats_df['Mean_Finish_Category_Checkpoint'] = pd.to_timedelta(checkpoints_stats_df['Mean_Finish_Category_Checkpoint_seconds'], unit='s')


checkpoints_stats_df[['Finish Category','Checkpoint','Stage', 'Mean_Finish_Category_Checkpoint','Median_Finish_Category_Checkpoint']][(checkpoints_stats_df['Finish Category'] == 'Sub-190') &
                                                                                                                                       (checkpoints_stats_df['Checkpoint'] != 'START')]

C:\Users\Karina\AppData\Local\Temp\ipykernel_14576\412525428.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merge['Duration_seconds'][df_merge['Duration_seconds']  <= 0] = np.nan


,Finish Category,Checkpoint,Stage,Mean_Finish_Category_Checkpoint,Median_Finish_Category_Checkpoint
72,Sub-190,Cogne IN,Stage 1,2 days 04:07:31,2 days 04:00:24
73,Sub-190,Cogne OUT,Time Spent in Cogne,0 days 03:23:37,0 days 03:20:22
74,Sub-190,Donnas IN,Stage 2,1 days 02:29:02,1 days 02:43:40
75,Sub-190,Donnas OUT,Time Spent in Donnas,0 days 02:18:11,0 days 01:56:51
77,Sub-190,Gressoney IN,Stage 3,1 days 03:00:32,1 days 03:19:54
78,Sub-190,Gressoney OUT,Time Spent in Gressoney,0 days 02:30:47,0 days 02:13:32
79,Sub-190,Rifugio Champillon,Stage 4,1 days 23:48:35,1 days 23:41:03
76,Sub-190,FINISH,Stage 4,0 days 22:59:37,0 days 23:08:13


### Who ran too easy or hard at the start?

In [40]:
sub_TOR450_dem_bib_list = list(TOR450_dem['PK'][TOR450_dem['Finish Category'] == 'Sub-160'].unique())

new_bib_list = []
for bib in sub_TOR450_dem_bib_list:
    df = lifebase_bib_df[
                ((lifebase_bib_df['Duration_hours']> 42) &
                (lifebase_bib_df['Lifebase'] == 'Cogne IN') )
    &
                (lifebase_bib_df['PK']== bib)
               ]
    
    new_bib_list.append(df)
df= pd.concat(new_bib_list)
pk_unique = list(df['PK'].unique())

for pk in pk_unique:
    print(lifebase_bib_df[['PK','Lifebase', 'Timestamp', 'Duration_hours']][(lifebase_bib_df['PK'] == pk)  ], '\n', '*'*40,  '\n',)
    print(TOR450_dem[['PK', 'Finish Category', 'Duration_hours']][(TOR450_dem['PK'] == pk)  ], '\n','\n','\n','\n', '*'*40,  '\n',)

                   PK       Lifebase           Timestamp  Duration_hours
680  TOR450_2022_4031          START 2022-09-09 20:00:00             NaN
681  TOR450_2022_4031       Cogne IN 2022-09-11 14:51:44       42.862222
682  TOR450_2022_4031      Cogne OUT 2022-09-11 15:49:52        0.968889
683  TOR450_2022_4031      Donnas IN 2022-09-12 17:49:27       25.993056
684  TOR450_2022_4031     Donnas OUT 2022-09-12 19:04:29        1.250556
685  TOR450_2022_4031   Gressoney IN 2022-09-13 18:30:57       23.441111
686  TOR450_2022_4031  Gressoney OUT 2022-09-13 19:42:29        1.192222
687  TOR450_2022_4031         FINISH 2022-09-16 07:35:50       59.889167 
 **************************************** 

                  PK Finish Category  Duration_hours
11  TOR450_2022_4031         Sub-160      155.597222 
 
 
 
 **************************************** 

                   PK       Lifebase           Timestamp  Duration_hours
840  TOR450_2022_4053          START 2022-09-09 20:00:00            

                   PK Finish Category  Duration_hours
509  TOR450_2021_4006         Sub-160      155.115278 
 
 
 
 **************************************** 

                   PK       Lifebase           Timestamp  Duration_hours
176  TOR450_2021_4027          START 2021-09-10 20:00:00             NaN
177  TOR450_2021_4027       Cogne IN 2021-09-12 16:39:13       44.653611
178  TOR450_2021_4027      Cogne OUT 2021-09-12 17:25:44        0.775278
179  TOR450_2021_4027      Donnas IN 2021-09-13 16:02:40       22.615556
180  TOR450_2021_4027     Donnas OUT 2021-09-13 17:18:28        1.263333
181  TOR450_2021_4027   Gressoney IN                 NaT             NaN
182  TOR450_2021_4027  Gressoney OUT 2021-09-14 19:17:35             NaN
183  TOR450_2021_4027         FINISH 2021-09-17 10:24:23       63.113333 
 **************************************** 

                   PK Finish Category  Duration_hours
510  TOR450_2021_4027         Sub-160      158.406389 
 
 
 
 ***********************

In [47]:
sub_TOR450_dem_bib_list = list(TOR450_dem['PK'][TOR450_dem['Finish Category'] == 'Sub-180'].unique())

new_bib_list = []
for bib in sub_TOR450_dem_bib_list:
    df = lifebase_bib_df[
                ((lifebase_bib_df['Duration_hours']< 50) &
                (lifebase_bib_df['Lifebase'] == 'Cogne IN') ) &
                (lifebase_bib_df['PK']== bib)]
    new_bib_list.append(df)
    
df= pd.concat(new_bib_list)

pk_unique = list(df['PK'].unique())

for pk in pk_unique:
    print(lifebase_bib_df[['PK',  'Lifebase', 'Duration_hours']][(lifebase_bib_df['PK'] == pk)  ], '\n', '\n',)
    print(TOR450_dem[['PK', 'Finish Category', 'Duration_hours']][(TOR450_dem['PK'] == pk)  ], '\n', '*'*40,  '\n',)

                   PK       Lifebase  Duration_hours
512  TOR450_2022_4010          START             NaN
513  TOR450_2022_4010       Cogne IN       46.647778
514  TOR450_2022_4010      Cogne OUT        1.808056
515  TOR450_2022_4010      Donnas IN       24.323056
516  TOR450_2022_4010     Donnas OUT        4.277222
517  TOR450_2022_4010   Gressoney IN       21.546667
518  TOR450_2022_4010  Gressoney OUT        4.308333
519  TOR450_2022_4010         FINISH       69.338889 
 

                  PK Finish Category  Duration_hours
31  TOR450_2022_4010         Sub-180          172.25 
 **************************************** 

                    PK       Lifebase  Duration_hours
1352  TOR450_2022_4124          START             NaN
1353  TOR450_2022_4124       Cogne IN       47.401389
1354  TOR450_2022_4124      Cogne OUT        4.557500
1355  TOR450_2022_4124      Donnas IN       23.230556
1356  TOR450_2022_4124     Donnas OUT        6.508333
1357  TOR450_2022_4124   Gressoney IN       

                   PK Finish Category  Duration_hours
534  TOR450_2021_4004         Sub-180      174.772222 
 **************************************** 



In [ ]:
# # reading in Raw Data
# races = ['TOR450']
# years = [ 
# #     '2021',
# #         '2022',
# #          '2023', 
#     '2024'
#         ]

# TORX_df = {}

# for race in races:
#     for year in years:
#         df = pd.read_excel(f'{race} Data/1. 100x100trail/{race}_{year}.xlsx',
#                                  dtype={'Start Date': 'string',
#                                         'Year': 'string'})
#         print(f'{race}_{year} {df.shape}')
#         # Store the DataFrame in the dictionary with a key like 'TOR450_2021'
#         TORX_df[f'{race}_{year}'] = df
#     print('*'*50)
    
# TORX_df_concat = pd.concat(TORX_df)
# TOR450 = TORX_df_concat[TORX_df_concat['Year'] == year]

In [ ]:
# sub_TOR450_dem_bib_list = list(TOR450_dem['PK'][TOR450_dem['Finish Category'] == 'Sub-90'].unique())

# new_bib_list = []
# for bib in sub_TOR450_dem_bib_list:
#     df = lifebase_bib_df[
#                 ((lifebase_bib_df['Duration_hours']> 8) &
#                 (lifebase_bib_df['Lifebase'] == 'Valgrisenche OUT') ) &
#                 (lifebase_bib_df['PK']== bib)]
#     new_bib_list.append(df)
    
# df= pd.concat(new_bib_list)

# pk_unique = list(df['PK'].unique())

# for pk in pk_unique:
#     print(lifebase_bib_df[['PK', 'Wave', 'Lifebase', 'Duration_hours']][(lifebase_bib_df['PK'] == pk)  ], '\n', '\n',)
#     print(TOR450_dem[['PK', 'Finish Category', 'Duration_hours']][(TOR450_dem['PK'] == pk)  ], '\n', '*'*40,  '\n',)